# DecodeLabs Data Analytics Project 1
## Data Cleaning & Preparation
**Dataset:** E-Commerce Orders Dataset

### Step 0: Load the Dataset

In [1]:
import pandas as pd
import numpy as np
import os

# Load dataset using relative path
file_path = "Dataset for Data Analytics - Sheet1.csv"
df = pd.read_csv(file_path)

print(f"Rows   : {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
print("\nFirst 5 Rows:")
df.head()

Rows   : 1200
Columns: 14

First 5 Rows:


,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04


### Data Types & Summary Statistics

In [2]:
print("Data Types:")
print(df.dtypes)

print("\nBasic Statistics:")
df.describe()

Data Types:
OrderID                str
Date                   str
CustomerID             str
Product                str
Quantity             int64
UnitPrice          float64
ShippingAddress        str
PaymentMethod          str
OrderStatus            str
TrackingNumber         str
ItemsInCart          int64
CouponCode             str
ReferralSource         str
TotalPrice         float64
dtype: object

Basic Statistics:


,Quantity,UnitPrice,ItemsInCart,TotalPrice
count,1200.000000,1200.000000,1200.000000,1200.000000
mean,2.945833,356.412750,5.485000,1053.968300
std,1.407557,197.177146,2.281983,819.856558
min,1.000000,11.390000,1.000000,11.390000
25%,2.000000,186.062500,4.000000,410.520000
50%,3.000000,364.210000,5.000000,823.615000
75%,4.000000,521.570000,7.000000,1578.475000
max,5.000000,699.930000,10.000000,3456.400000


### Step 1: Identify Missing / Null Values

In [3]:
missing_counts = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'Column': missing_counts.index,
    'Missing Count': missing_counts.values,
    'Missing %': missing_pct.values
})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

total_missing = df.isnull().sum().sum()
total_cells = df.shape[0] * df.shape[1]

print(f"Total cells in dataset  : {total_cells}")
print(f"Total missing values    : {total_missing}")
print(f"Overall missing %       : {(total_missing / total_cells * 100):.2f}%\n")

if len(missing_df) > 0:
    display(missing_df)
else:
    print("No missing values found in any column!")

Total cells in dataset  : 16800
Total missing values    : 0
Overall missing %       : 0.00%

No missing values found in any column!


### Step 1b: Handle Missing Values

In [4]:
for col in df.columns:
    missing = df[col].isnull().sum()
    if missing > 0:
        if col == 'CouponCode':
            df[col] = df[col].fillna('NoCoupon')
            print(f"[OK] '{col}': Filled {missing} missing values with 'NoCoupon'")
        elif df[col].dtype in ['float64', 'int64']:
            median_val = df[col].median()
            df[col] = df[col].fillna(median_val)
            print(f"[OK] '{col}': Filled {missing} missing values with median ({median_val})")
        else:
            mode_val = df[col].mode()[0]
            df[col] = df[col].fillna(mode_val)
            print(f"[OK] '{col}': Filled {missing} missing values with mode ('{mode_val}')")

print(f"\nMissing values remaining: {df.isnull().sum().sum()}")


Missing values remaining: 0


### Step 2: Remove Duplicate Records

In [5]:
full_dupes = df.duplicated().sum()
oid_dupes = df.duplicated(subset=['OrderID']).sum()
cid_count = df['CustomerID'].nunique()

print(f"Fully Duplicate Rows: {full_dupes}")
print(f"Duplicate OrderIDs   : {oid_dupes}")
print(f"Unique Customers     : {cid_count} out of {len(df)} orders")

if full_dupes > 0:
    df.drop_duplicates(inplace=True)
    print(f"[OK] After removal of full duplicates: {len(df)} rows remain.")

if oid_dupes > 0:
    df.drop_duplicates(subset=['OrderID'], keep='first', inplace=True)
    print(f"[OK] After removal of duplicate OrderIDs: {len(df)} rows remain.")
else:
    print("All OrderIDs are unique.")

Fully Duplicate Rows: 0
Duplicate OrderIDs   : 0
Unique Customers     : 1189 out of 1200 orders
All OrderIDs are unique.


### Step 3: Correct Data Formats

In [6]:
# 3a. Date Format Validation
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
invalid_dates = df['Date'].isnull().sum()

if invalid_dates > 0:
    df.dropna(subset=['Date'], inplace=True)
    print(f"Dropped {invalid_dates} rows with invalid dates.")
else:
    print("All dates are valid and correctly formatted.")

date_range = f"{df['Date'].min().strftime('%Y-%m-%d')} to {df['Date'].max().strftime('%Y-%m-%d')}"
print(f"Date range: {date_range}")

All dates are valid and correctly formatted.
Date range: 2023-01-01 to 2025-06-30


In [7]:
# 3b. Numeric Validation & TotalPrice Verification
numeric_cols = ['Quantity', 'UnitPrice', 'ItemsInCart', 'TotalPrice']
for col in numeric_cols:
    negatives = (df[col] < 0).sum()
    if negatives > 0:
        print(f"WARNING: '{col}' has {negatives} negative values!")

df['Calculated_Total'] = df['Quantity'] * df['UnitPrice']
df['Price_Diff'] = abs(df['TotalPrice'] - df['Calculated_Total'])
price_mismatches = (df['Price_Diff'] > 0.01).sum()

if price_mismatches > 0:
    print(f"Correcting {price_mismatches} TotalPrice mismatches...")
    df['TotalPrice'] = df['Calculated_Total']
else:
    print("All TotalPrice values are consistent.")

df.drop(columns=['Calculated_Total', 'Price_Diff'], inplace=True)

All TotalPrice values are consistent.


In [8]:
# 3c. Categorical & ID Validation
cat_cols = ['Product', 'PaymentMethod', 'OrderStatus', 'CouponCode', 'ReferralSource']
for col in cat_cols:
    if df[col].dtype == 'object':
        df[col] = df[col].str.strip()

oid_pattern = df['OrderID'].str.match(r'^ORD\\d{6}$')
trk_pattern = df['TrackingNumber'].str.match(r'^TRK\\d{8}$')

print(f"Valid OrderIDs      : {oid_pattern.sum()} / {len(df)}")
print(f"Valid TrackingNumbers: {trk_pattern.sum()} / {len(df)}")

Valid OrderIDs      : 0 / 1200
Valid TrackingNumbers: 0 / 1200


### Step 4: Data Quality Summary (Post-Cleaning)

In [9]:
print(f"Final Dataset Shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Missing Values     : {df.isnull().sum().sum()}")
print(f"Duplicate Rows     : {df.duplicated().sum()}")
print(f"Duplicate OrderIDs : {df.duplicated(subset=['OrderID']).sum()}")

df.info()

Final Dataset Shape: 1200 rows x 14 columns
Missing Values     : 0
Duplicate Rows     : 0
Duplicate OrderIDs : 0
<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   OrderID          1200 non-null   str           
 1   Date             1200 non-null   datetime64[us]
 2   CustomerID       1200 non-null   str           
 3   Product          1200 non-null   str           
 4   Quantity         1200 non-null   int64         
 5   UnitPrice        1200 non-null   float64       
 6   ShippingAddress  1200 non-null   str           
 7   PaymentMethod    1200 non-null   str           
 8   OrderStatus      1200 non-null   str           
 9   TrackingNumber   1200 non-null   str           
 10  ItemsInCart      1200 non-null   int64         
 11  CouponCode       1200 non-null   str           
 12  ReferralSource   1200 non-null   str        

### Step 5: Exploratory Data Analysis (EDA)

In [10]:
# 5a. Product Distribution & Revenue
product_counts = df['Product'].value_counts()
product_revenue = df.groupby('Product')['TotalPrice'].sum()
product_avg_price = df.groupby('Product')['UnitPrice'].mean().round(2)

product_summary = pd.DataFrame({
    'Order Count': product_counts,
    'Order %': (product_counts / len(df) * 100).round(1),
    'Total Revenue': product_revenue,
    'Avg Unit Price': product_avg_price
}).sort_values('Total Revenue', ascending=False)

product_summary

,Order Count,Order %,Total Revenue,Avg Unit Price
Product,,,,
Chair,178,14.8,195620.11,355.66
Printer,181,15.1,195612.61,351.71
Laptop,173,14.4,192126.56,357.71
Tablet,179,14.9,186568.95,367.68
Monitor,163,13.6,175651.41,358.66
Desk,170,14.2,167459.93,329.61
Phone,156,13.0,151722.39,375.22


In [11]:
# 5b. Order Status Breakdown
status_summary = df.groupby('OrderStatus').agg(
    Orders=('OrderID', 'count'),
    Revenue=('TotalPrice', 'sum')
)
status_summary['Order %'] = (status_summary['Orders'] / len(df) * 100).round(1)
status_summary.sort_values('Orders', ascending=False)

,Orders,Revenue,Order %
OrderStatus,,,
Cancelled,250,276396.21,20.8
Returned,247,243277.70,20.6
Pending,237,256328.15,19.8
Shipped,235,246159.58,19.6
Delivered,231,242600.32,19.2


In [12]:
# 5c & 5d. Payment Method & Coupon Code Analysis
payment_summary = df.groupby('PaymentMethod').agg(
    Orders=('OrderID', 'count'),
    TotalRevenue=('TotalPrice', 'sum'),
    AvgOrderValue=('TotalPrice', 'mean')
).round(2).sort_values('TotalRevenue', ascending=False)

coupon_summary = df.groupby('CouponCode').agg(
    Orders=('OrderID', 'count'),
    TotalRevenue=('TotalPrice', 'sum'),
    AvgOrderValue=('TotalPrice', 'mean')
).round(2).sort_values('TotalRevenue', ascending=False)

print("Payment Method Summary:")
display(payment_summary)

print("\nCoupon Code Summary:")
display(coupon_summary)

Payment Method Summary:


,Orders,TotalRevenue,AvgOrderValue
PaymentMethod,,,
Credit Card,234,263847.63,1127.55
Online,258,262442.94,1017.22
Cash,246,259786.29,1056.04
Gift Card,230,246323.92,1070.97
Debit Card,232,232361.18,1001.56



Coupon Code Summary:


,Orders,TotalRevenue,AvgOrderValue
CouponCode,,,
FREESHIP,313,335036.99,1070.41
NoCoupon,309,322401.41,1043.37
SAVE10,286,304840.02,1065.87
WINTER15,292,302483.54,1035.90


In [13]:
# 5e & 5f. Referral Source & Yearly Trends
ref_summary = df.groupby('ReferralSource').agg(
    Orders=('OrderID', 'count'),
    TotalRevenue=('TotalPrice', 'sum'),
    AvgOrderValue=('TotalPrice', 'mean')
).round(2).sort_values('TotalRevenue', ascending=False)

df['Year'] = df['Date'].dt.year
yearly_summary = df.groupby('Year').agg(
    Orders=('OrderID', 'count'),
    Revenue=('TotalPrice', 'sum'),
    AvgOrderValue=('TotalPrice', 'mean')
).round(2)

print("Referral Source Summary:")
display(ref_summary)

print("\nYearly Trend Summary:")
display(yearly_summary)

Referral Source Summary:


,Orders,TotalRevenue,AvgOrderValue
ReferralSource,,,
Instagram,259,275285.45,1062.88
Email,250,261808.55,1047.23
Google,241,250441.48,1039.18
Facebook,228,250410.90,1098.29
Referral,222,226815.58,1021.69



Yearly Trend Summary:


,Orders,Revenue,AvgOrderValue
Year,,,
2023,510,552643.24,1083.61
2024,459,480235.87,1046.27
2025,231,231882.85,1003.82


### Step 6: Save Cleaned Dataset

In [14]:
if 'Year' in df.columns:
    df.drop(columns=['Year'], inplace=True)

output_path = "Cleaned_Dataset.csv"
df.to_csv(output_path, index=False)
print(f"Cleaned dataset saved successfully to '{output_path}'")

Cleaned dataset saved successfully to 'Cleaned_Dataset.csv'
